In [ ]:
import mg5qs_imports as qs
from pathlib import Path
import os
import numpy as np

### Useful inquiry; changing parameters automatically

For many intresting inquarys, it is nessary to change parameters between runs, within the same framework. For example, we may want to varry matrix ellements which encode the strength of some NP interaction between generation of LHEs. This requires two new features: first, we must edit the param_card automatically (not using visual editing tools); and second, we must ascocate the parameters to their respective LHEs.

This example includes:
- Running over a space of parameters
- Using ParamCard API to script generation of LHEs
- Ascocating results with parameters
- Preforming statistical comparisons with standard libraries

### Produce a MadGraph framework 

Omitting the proc_card and run_card visualizations...

In [ ]:
INPUT_PATH = Path.cwd()/'mg5' # madgraph cards
output_name, FRAMEWORK_PATH = qs.run_MG5(INPUT_PATH, proc_card_name='proc_card_space.dat')

### Use ParamCard object

To read and write the param_card. First, we will declare a ParamCard object and explore some of its capeabilties. 

In [ ]:
card = qs.ParamCard(FRAMEWORK_PATH)
card #echoing a ParamCard displays its member 'blocks'

Each of the above blocks stores useful parameters. Noteably, if a new physics (NP) model is loaded, its coresponding blocks will also display here. This means everything discussed here works also for NP models.

For example, we can look at the underlying pandas dataframe which represents the **MASS** block:

In [ ]:
card.df('MASS')

If instead we are intrested in a single value from this dataframe, we can use get_value. This is the prefered way to interact with ParamCards. 

In [ ]:
tau_id = 15
card.get_value('MASS', tau_id)

As you may have suspected, to round out the functionallity, there is also a set_value command.

In [ ]:
card.set_value('MASS', tau_id, 100)
card.get_value('MASS', tau_id)

### Generate LHEs while verrying parameters

Once the card value is set, it will need to be passed as an argument when generating LHEs.

So, to very over a space of parmeters all we need is a simple loop.  

In [ ]:
tau_id = 15
card = qs.ParamCard(FRAMEWORK_PATH)
MTAU = np.linspace(1.777-1, 1.777+1, 9)

for mtau in MTAU:
    card.set_value('MASS', tau_id, mtau)
    qs.generate_LHE(card, FRAMEWORK_PATH)

### Shower 

In [ ]:
qs.pythia_parallel([tau_id], FRAMEWORK_PATH, 'EXAMPLE_SPACE', topics='P_mu', size=100000)

### Load data

Using **concat=False** to respect the fact that LHEs were generated with different parameters. 

In [ ]:
data = qs.unpickle('EXAMPLE_SPACE', concat=False)

In [ ]:
import scipy.stats
import pandas as pd
import matplotlib.pyplot as plt

When loading data in concat=False mode, the return value will look like this schematic: **dict[unique key name] = (ParamCard, dataframe)**.

Let's practice traverseing one before trying to ascocate data with its respective parameter or drawing a plot. 

In [ ]:
data.keys() # notice that the keys are (very probably) out of order

In [ ]:
param_card, df = data['EXAMPLE_SPACE_0'] # python will unpack return values 

print(type(param_card), type(df))

In [ ]:
param_card

In [ ]:
df

### Compute custom statistics & ascocate run parameters

Now, the process to compute a new column will require looping over all dataframes. Since they are stored indevdually in the values of the values of the dictionary.

In [ ]:
for v in data.values(): # loop over dataframes
    v[1]['pT'] = np.sqrt((v[1]['px']**2)+(v[1]['py']**2)) 

Since it is often useful to compute statistics with respect to the SM run, we need to first ascocate runs with their mass paremeters, then find which is the SM. 

In [ ]:
TAU_ID = 15
PARAMCARD, DF = 0, 1

mass_key = [(data[k][PARAMCARD].get_value('MASS', TAU_ID), k) for k in data.keys()]
mass_key # a tuple containg (tau mass, key)

Now we must sort by $m_\tau$.

In [ ]:
mass_key = sorted(mass_key, key=lambda x: x[0])
mass_key # notice that the LHS is sorteted and the RHS is scrambled

In [ ]:
# find where m_tau = m_tau,SM
for i,mk in enumerate(mass_key):
    if np.isclose(mk[0], 1.777):
        SM_INDX = i
        break
SM_INDX

### Use SciPy to compute statistical comparison

From here, you could do any inquary. In this case, I will preform a KS test between the standard model and each other run. This will be highly unstable because we do not have enough data to overcome noise, but it is a good ilistration. 

In [ ]:
pd.Series([scipy.stats.ks_2samp(data[mass_key[SM_INDX][1]][DF]['pT'], data[m_k[1]][DF]['pT']).pvalue for m_k in mass_key])

### Plot distributions

In simular fashion to above.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(12, 10))
axes = axes.flatten()
BINS = np.linspace(0, 200, 20)

for idx, m_k in enumerate(mass_key):
    ax = axes[idx]
    ax.hist(data[m_k[1]][1]['pT'], bins=BINS)
    ax.set_title(f"$m_\\tau$ = {m_k[0]}")
    ax.set_xlabel('GeV')
    ax.set_yscale('log')

fig.suptitle(f"$p_T$ distribution for verous $\\tau$ masses", fontsize=18)
plt.tight_layout()
plt.show()